# **Proyecto 2: Embeddings y Búsqueda Semántica**

**Curso:** CC0C2 - Procesamiento del Lenguaje Natural  
**Estudiante:** César (CC0C2 Estudiante)  
**Cuaderno Base:** [Cuaderno 21 - CC0C2](file:///Users/cesar/Desktop/CC-0C2/Semana10/Cuaderno21-CC0C2.ipynb)  

Este notebook presenta la implementación de un sistema de recuperación de información híbrido que combina la búsqueda léxica basada en frecuencia de términos (BM25) con la búsqueda semántica densa utilizando dos modelos de embeddings preentrenados diferentes. Evaluamos el impacto del modelo y de la métrica de distancia en la calidad de recuperación.

In [2]:
# CELDA DE VERIFICACIÓN PERSONAL
STUDENT_NAME = "César (CC0C2 Estudiante)"
EXECUTION_DATE = "2026-06-28"
BASE_NOTEBOOK = "Cuaderno21-CC0C2.ipynb"
MODEL_NAME = "BAAI/bge-small-en-v1.5 & all-MiniLM-L6-v2"
SEED = 42
VARIANT = "Aislamiento de dimensiones (d=384) con BGE-small vs. all-MiniLM-L6-v2 y evaluación híbrida"

print("Estudiante:", STUDENT_NAME)
print("Fecha:", EXECUTION_DATE)
print("Cuaderno base:", BASE_NOTEBOOK)
print("Modelos:", MODEL_NAME)
print("Semilla:", SEED)
print("Variante:", VARIANT)

Estudiante: César (CC0C2 Estudiante)
Fecha: 2026-06-28
Cuaderno base: Cuaderno21-CC0C2.ipynb
Modelos: BAAI/bge-small-en-v1.5 & all-MiniLM-L6-v2
Semilla: 42
Variante: Aislamiento de dimensiones (d=384) con BGE-small vs. all-MiniLM-L6-v2 y evaluación híbrida


## **1. Objetivo del Proyecto e Ingesta de Datos**

### **Objetivo**
El objetivo principal es implementar y comparar un pipeline de búsqueda de información léxica (BM25) y semántica (vectores densos) sobre un corpus especializado de la película *Interstellar*. Evaluamos la influencia del modelo de embeddings (`BAAI/bge-small-en-v1.5` vs. `sentence-transformers/all-MiniLM-L6-v2`) y de las métricas de distancia (Similitud Coseno vs. Distancia Euclidiana $L_2$).

### **Marco Teórico**
1. **Búsqueda Léxica (BM25)**: Es una extensión probabilística del modelo TF-IDF. Calcula el score de un documento basado en la coincidencia exacta de términos en la consulta, saturando la frecuencia del término e incorporando una penalización según el tamaño del documento respecto al tamaño medio:
   $$\text{Score}_{\text{BM25}}(D, Q) = \sum_{q \in Q} \text{IDF}(q) \cdot \frac{f(q, D) \cdot (k_1 + 1)}{f(q, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{L_{\text{avg}}}\right)}$$
   donde $k_1$ controla la saturación del término y $b$ controla la penalización de longitud.

2. **Búsqueda Semántica Densa**: Mapea la consulta y los documentos en un espacio vectorial continuo de dimensión $d$ tal que el producto interno (o coseno) entre los vectores represente el grado de similitud semántica. 
   - **Similitud Coseno**: Mide el coseno del ángulo entre los dos vectores. Es equivalente al producto interno cuando ambos vectores están normalizados a norma unidad ($L_2$):
     $$\cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \|\vec{B}\|}$$
   - **Distancia Euclidiana ($L_2$)**: Mide la distancia geométrica directa en el espacio Euclidiano:
     $$d_{L_2}(\vec{A}, \vec{B}) = \sqrt{\sum_{i=1}^d (A_i - B_i)^2}$$

In [3]:
import numpy as np
import pandas as pd
import faiss
import string
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# 1. Definición del Corpus (15 documentos enriquecidos sobre Interstellar)
corpus = [
    "Interstellar is a science fiction film directed by Christopher Nolan. It explores themes of space travel, black holes, wormholes, and the survival of humanity in the face of global crop blight.",
    "The movie's scientific advisor, theoretical physicist Kip Thorne, worked on the physics of the black hole Gargantua and wormholes to ensure they were scientifically accurate and realistic.",
    "The plot centers on Cooper, a former NASA pilot who is recruited for a mission to pilot the spaceship Endurance through a wormhole near Saturn in search of a new home for mankind.",
    "Gargantua is the massive, rapidly rotating black hole in Interstellar. Kip Thorne's equations were used to render its accretion disk, producing a visually stunning and scientifically accurate depiction of gravitational lensing.",
    "The emotional core of Interstellar revolves around the relationship between Cooper and his daughter Murph. Cooper's promise to return and the effects of gravitational time dilation shape their bond.",
    "On Miller's Planet, which orbits close to Gargantua, gravitational time dilation is extreme: one hour on the surface equals seven years back on Earth, leading to tragic consequences for the crew.",
    "The dust storms and agricultural blight on Earth have devastated food supplies. Corn is the last remaining crop, and humanity faces extinction within a generation if a new planet is not colonized.",
    "Plan A, developed by Professor Brand, involves solving a gravity equation to launch massive space stations to colonize new worlds. However, this requires data from inside a black hole's event horizon.",
    "Plan B is a population bomb scenario using frozen human embryos to colonize the most habitable planet if Plan A fails and Earth's population cannot be saved.",
    "Mann's Planet is an icy, inhospitable world with frozen clouds. Dr. Mann, driven mad by isolation, falsified data to make NASA believe the planet was habitable so they would rescue him.",
    "Edmunds' Planet is ultimately revealed to be the most habitable world, with breathable air and stable ground. Amelia Brand travels there at the end of the film to establish the new human colony.",
    "TARS and CASE are tactical robots characterized by their rectangular design and adjustable humor, honesty, and sarcasm parameters. They provide critical assistance during the mission.",
    "The tesseract, or five-dimensional space, is constructed by future humans inside Gargantua. It allows Cooper to communicate with Murph across time using gravity as a medium.",
    "Interstellar's soundtrack was composed by Hans Zimmer. It features a pipe organ and atmospheric melodies that emphasize the vastness of space and the intimacy of the father-daughter relationship.",
    "Critics praised Interstellar for its visual effects, acting, and ambition, though some found the plot complex and the resolution involving love and gravity somewhat sentimental."
]

print(f"Corpus ingestado. Total de documentos: {len(corpus)}")
for idx, text in enumerate(corpus):
    print(f"Doc {idx}: {text[:80]}...")

Corpus ingestado. Total de documentos: 15
Doc 0: Interstellar is a science fiction film directed by Christopher Nolan. It explore...
Doc 1: The movie's scientific advisor, theoretical physicist Kip Thorne, worked on the ...
Doc 2: The plot centers on Cooper, a former NASA pilot who is recruited for a mission t...
Doc 3: Gargantua is the massive, rapidly rotating black hole in Interstellar. Kip Thorn...
Doc 4: The emotional core of Interstellar revolves around the relationship between Coop...
Doc 5: On Miller's Planet, which orbits close to Gargantua, gravitational time dilation...
Doc 6: The dust storms and agricultural blight on Earth have devastated food supplies. ...
Doc 7: Plan A, developed by Professor Brand, involves solving a gravity equation to lau...
Doc 8: Plan B is a population bomb scenario using frozen human embryos to colonize the ...
Doc 9: Mann's Planet is an icy, inhospitable world with frozen clouds. Dr. Mann, driven...
Doc 10: Edmunds' Planet is ultimately revealed t

In [4]:
# 2. Búsqueda Léxica con BM25
def tokenize_text(text):
    # Remover puntuación y convertir a minúsculas
    text_cleaned = text.translate(str.maketrans('', '', string.punctuation)).lower()
    return text_cleaned.split()

tokenized_corpus = [tokenize_text(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def search_bm25(query: str, k: int = 3):
    tokenized_query = tokenize_text(query)
    scores = bm25.get_scores(tokenized_query)
    # Ordenar índices descendente
    ranked_idx = np.argsort(scores)[::-1]
    
    results = []
    for idx in ranked_idx[:k]:
        results.append({
            "index": int(idx),
            "score": float(scores[idx]),
            "text": corpus[idx]
        })
    return results

# Prueba rápida
print("Prueba BM25 para 'Kip Thorne physics':")
for res in search_bm25("Kip Thorne physics", k=2):
    print(f"  Doc {res['index']} (score={res['score']:.4f}): {res['text'][:100]}")

Prueba BM25 para 'Kip Thorne physics':
  Doc 1 (score=6.4548): The movie's scientific advisor, theoretical physicist Kip Thorne, worked on the physics of the black
  Doc 3 (score=1.6444): Gargantua is the massive, rapidly rotating black hole in Interstellar. Kip Thorne's equations were u


In [5]:
# 3. Modelo A (Línea Base) - BAAI/bge-small-en-v1.5 (Dimensión d = 384)
model_a_name = "BAAI/bge-small-en-v1.5"
embedder_a = SentenceTransformer(model_a_name)

# Generación de embeddings para el corpus
embeddings_a_norm = embedder_a.encode(corpus, convert_to_numpy=True, normalize_embeddings=True) # Para Similitud Coseno / Inner Product
embeddings_a_raw = embedder_a.encode(corpus, convert_to_numpy=True, normalize_embeddings=False) # Para L2

dim = embeddings_a_norm.shape[1]
print(f"Modelo A ({model_a_name}) cargado.")
print(f"Dimensiones de los embeddings: {dim}")

# Construcción de índices FAISS
# Índice de Producto Interno (equivalente a Coseno por estar normalizados)
index_a_ip = faiss.IndexFlatIP(dim)
index_a_ip.add(embeddings_a_norm.astype("float32"))

# Índice L2
index_a_l2 = faiss.IndexFlatL2(dim)
index_a_l2.add(embeddings_a_raw.astype("float32"))

def search_dense_a(query: str, metric: str = "cosine", k: int = 3):
    if metric == "cosine":
        # Consulta normalizada para Coseno
        q_vec = embedder_a.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        scores, indices = index_a_ip.search(q_vec, k)
    else:
        # Consulta sin normalizar para L2
        q_vec = embedder_a.encode([query], convert_to_numpy=True, normalize_embeddings=False).astype("float32")
        scores, indices = index_a_l2.search(q_vec, k)
        
    results = []
    for s, idx in zip(scores[0], indices[0]):
        results.append({
            "index": int(idx),
            "score": float(s),
            "text": corpus[idx]
        })
    return results

# Prueba rápida
print("Prueba Coseno Modelo A:")
for res in search_dense_a("Kip Thorne physics", metric="cosine", k=2):
    print(f"  Doc {res['index']} (similitud={res['score']:.4f}): {res['text'][:100]}")

Modelo A (BAAI/bge-small-en-v1.5) cargado.
Dimensiones de los embeddings: 384
Prueba Coseno Modelo A:
  Doc 1 (similitud=0.7114): The movie's scientific advisor, theoretical physicist Kip Thorne, worked on the physics of the black
  Doc 3 (similitud=0.6573): Gargantua is the massive, rapidly rotating black hole in Interstellar. Kip Thorne's equations were u


In [ ]:
# 1. Cálculo manual con NumPy
q_emb = embedder_a.encode("Kip Thorne physics", normalize_embeddings=False)
sims = manual_cosine_similarity(q_emb, embeddings_a_raw)

# 2. Imprimimos el Top-3 manual
for idx in np.argsort(sims)[::-1][:3]:
    print(f"Manual -> Doc {idx} | Score: {sims[idx]:.4f}")

# 3. Comparamos contra la búsqueda FAISS
print("\nFAISS:")
for res in search_dense_a("Kip Thorne physics", metric="cosine", k=3):
    print(f"FAISS  -> Doc {res['index']} | Score: {res['score']:.4f}")


NameError: name 'c' is not defined

In [6]:
# 4. Modelo B (Variante) - sentence-transformers/all-MiniLM-L6-v2 (Dimensión d = 384)
model_b_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder_b = SentenceTransformer(model_b_name)

# Generación de embeddings para el corpus
embeddings_b_norm = embedder_b.encode(corpus, convert_to_numpy=True, normalize_embeddings=True)
embeddings_b_raw = embedder_b.encode(corpus, convert_to_numpy=True, normalize_embeddings=False)

print(f"Modelo B ({model_b_name}) cargado.")
print(f"Dimensiones de los embeddings: {embeddings_b_norm.shape[1]}")

# Construcción de índices FAISS para Modelo B
index_b_ip = faiss.IndexFlatIP(dim)
index_b_ip.add(embeddings_b_norm.astype("float32"))

index_b_l2 = faiss.IndexFlatL2(dim)
index_b_l2.add(embeddings_b_raw.astype("float32"))

def search_dense_b(query: str, metric: str = "cosine", k: int = 3):
    if metric == "cosine":
        q_vec = embedder_b.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        scores, indices = index_b_ip.search(q_vec, k)
    else:
        q_vec = embedder_b.encode([query], convert_to_numpy=True, normalize_embeddings=False).astype("float32")
        scores, indices = index_b_l2.search(q_vec, k)
        
    results = []
    for s, idx in zip(scores[0], indices[0]):
        results.append({
            "index": int(idx),
            "score": float(s),
            "text": corpus[idx]
        })
    return results

# Prueba rápida
print("Prueba Coseno Modelo B:")
for res in search_dense_b("Kip Thorne physics", metric="cosine", k=2):
    print(f"  Doc {res['index']} (similitud={res['score']:.4f}): {res['text']}")

/Users/cesar/Desktop/CC-0C2/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Modelo B (sentence-transformers/all-MiniLM-L6-v2) cargado.
Dimensiones de los embeddings: 384
Prueba Coseno Modelo B:
  Doc 1 (similitud=0.5942): The movie's scientific advisor, theoretical physicist Kip Thorne, worked on the physics of the black hole Gargantua and wormholes to ensure they were scientifically accurate and realistic.
  Doc 3 (similitud=0.3466): Gargantua is the massive, rapidly rotating black hole in Interstellar. Kip Thorne's equations were used to render its accretion disk, producing a visually stunning and scientifically accurate depiction of gravitational lensing.


In [7]:
# 5. Búsqueda Híbrida con Normalización de Scores
def min_max_normalize(scores):
    scores = np.array(scores, dtype="float32")
    s_min, s_max = scores.min(), scores.max()
    if s_min == s_max:
        return np.ones_like(scores)
    return (scores - s_min) / (s_max - s_min)

def search_hybrid(query: str, model_choice: str = "A", k: int = 3, alpha: float = 0.5):
    # 1. Obtener scores léxicos
    tokenized_query = tokenize_text(query)
    sparse_scores = np.array(bm25.get_scores(tokenized_query), dtype="float32")
    
    # 2. Obtener scores densos sobre todo el corpus
    if model_choice == "A":
        q_vec = embedder_a.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        dense_scores, dense_indices = index_a_ip.search(q_vec, len(corpus))
    else:
        q_vec = embedder_b.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        dense_scores, dense_indices = index_b_ip.search(q_vec, len(corpus))
        
    # Alinear scores densos con los índices del corpus
    # index.search devuelve (scores, indices). Los ordenamos para alinearlos con el orden original [0, 1, ..., N-1]
    indices = dense_scores[0] # índices devueltos
    values = dense_scores[0]   # scores devueltos
    
    # Reorganizar los scores densos para que correspondan al índice original del corpus
    # Para ello, index.search sobre todo el corpus devuelve todos los índices en algún orden
    dense_scores_aligned = np.zeros(len(corpus), dtype="float32")
    for val, idx in zip(dense_scores[0], dense_indices[0]):
        dense_scores_aligned[idx] = val
        
    # 3. Normalizar ambos scores
    sparse_norm = min_max_normalize(sparse_scores)
    dense_norm = min_max_normalize(dense_scores_aligned)
    
    # 4. Fusionar linealmente
    fused_scores = alpha * dense_norm + (1.0 - alpha) * sparse_norm
    ranked_idx = np.argsort(fused_scores)[::-1]
    
    results = []
    for idx in ranked_idx[:k]:
        results.append({
            "index": int(idx),
            "score": float(fused_scores[idx]),
            "text": corpus[idx]
        })
    return results

# Prueba rápida
print("Prueba Híbrida:")
for res in search_hybrid("Kip Thorne physics", model_choice="A", k=2, alpha=0.5):
    print(f"  Doc {res['index']} (fused_score={res['score']:.4f}): {res['text']}")

Prueba Híbrida:
  Doc 1 (fused_score=1.0000): The movie's scientific advisor, theoretical physicist Kip Thorne, worked on the physics of the black hole Gargantua and wormholes to ensure they were scientifically accurate and realistic.
  Doc 3 (fused_score=0.5336): Gargantua is the massive, rapidly rotating black hole in Interstellar. Kip Thorne's equations were used to render its accretion disk, producing a visually stunning and scientifically accurate depiction of gravitational lensing.


In [8]:
# 6. Configuración Experimental y Benchmark de Evaluación

# Dataset de Prueba: 5 consultas con sus documentos relevantes (gold standards)
benchmark_data = [
    {
        "query": "How accurate is the rotating black hole Gargantua physics?",
        "gold_indices": {3, 1, 5}, # Gargantua (doc 3), Kip Thorne (doc 1), Miller's Planet near Gargantua (doc 5)
        "reference": "The depiction of Gargantua was highly accurate because Kip Thorne's physical equations were used to render the accretion disk and gravitational lensing."
    },
    {
        "query": "Cooper and Murph daughter family bond and relationship",
        "gold_indices": {4, 13}, # Relationship (doc 4), Tesseract/gravity communication (doc 13)
        "reference": "The emotional core is the relationship between Cooper and his daughter Murph, which is maintained across space and time using gravity inside the tesseract."
    },
    {
        "query": "dust storms blight and food corn crop extinction on Earth",
        "gold_indices": {6, 0}, # Earth crop crisis (doc 6), global crop blight (doc 0)
        "reference": "Earth is devastated by dust storms and crop blight. Corn is the last remaining crop, and humanity faces extinction if they do not colonize a new planet."
    },
    {
        "query": "Endurance mission wormhole near Saturn space travel",
        "gold_indices": {2, 0}, # Endurance wormhole (doc 2), space travel themes (doc 0)
        "reference": "Cooper joins a space mission in the Endurance, travelling through a wormhole near Saturn in search of habitable planets."
    },
    {
        "query": "Hans Zimmer organ music soundtrack score",
        "gold_indices": {13}, # Zimmer soundtrack (doc 13 is index 13 in 0-indexed: Hans Zimmer pipe organ)
        "reference": "The film's atmospheric score was composed by Hans Zimmer, highlighting the pipe organ to represent the vastness of space and family intimacy."
    }
]

def calculate_recall_at_k(retrieved_indices, gold_indices):
    # Recall@k: ¿Qué porcentaje de los documentos relevantes fueron recuperados?
    intersection = set(retrieved_indices) & set(gold_indices)
    return len(intersection) / len(gold_indices)

def calculate_mrr_at_k(retrieved_indices, gold_indices):
    # MRR: Recíproco de la posición del primer documento relevante
    for rank, idx in enumerate(retrieved_indices, start=1):
        if idx in gold_indices:
            return 1.0 / rank
    return 0.0

def run_benchmark(retriever_func, k=3):
    recalls = []
    mrrs = []
    for item in benchmark_data:
        results = retriever_func(item["query"], k=k)
        ret_indices = [r["index"] for r in results]
        recalls.append(calculate_recall_at_k(ret_indices, item["gold_indices"]))
        mrrs.append(calculate_mrr_at_k(ret_indices, item["gold_indices"]))
    return np.mean(recalls), np.mean(mrrs)

# Evaluamos las diferentes configuraciones
metrics = {}

# 1. BM25
metrics["BM25"] = run_benchmark(search_bm25, k=3)

# 2. Densa A (BGE-small) - Coseno
metrics["Dense A (BGE) - Coseno"] = run_benchmark(lambda q, k: search_dense_a(q, "cosine", k), k=3)

# 3. Densa A (BGE) - L2
metrics["Dense A (BGE) - L2"] = run_benchmark(lambda q, k: search_dense_a(q, "l2", k), k=3)

# 4. Densa B (MiniLM) - Coseno
metrics["Dense B (MiniLM) - Coseno"] = run_benchmark(lambda q, k: search_dense_b(q, "cosine", k), k=3)

# 5. Híbrido (Densa A + BM25, alpha=0.5)
metrics["Híbrido (BGE + BM25, a=0.5)"] = run_benchmark(lambda q, k: search_hybrid(q, "A", k, 0.5), k=3)

# 6. Híbrido (Densa B + BM25, alpha=0.5)
metrics["Híbrido (MiniLM + BM25, a=0.5)"] = run_benchmark(lambda q, k: search_hybrid(q, "B", k, 0.5), k=3)

# Presentamos los resultados en una tabla de Pandas
df_metrics = pd.DataFrame.from_dict(
    metrics, orient="index", columns=["Mean Recall@3", "Mean MRR@3"]
)
print("=== COMPARACIÓN DE RETRIEVERS (LÍNEA BASE vs. VARIANTES) ===")
display(df_metrics)

=== COMPARACIÓN DE RETRIEVERS (LÍNEA BASE vs. VARIANTES) ===


,Mean Recall@3,Mean MRR@3
BM25,1.000000,1.0
Dense A (BGE) - Coseno,0.733333,1.0
Dense A (BGE) - L2,0.733333,1.0
Dense B (MiniLM) - Coseno,0.733333,1.0
"Híbrido (BGE + BM25, a=0.5)",0.933333,1.0
"Híbrido (MiniLM + BM25, a=0.5)",1.000000,1.0


In [9]:
query = "How accurate is the rotating black hole Gargantua physics?"

# Usando el Modelo A (BGE-small) con Similitud Coseno
query_embedding = embedder_a.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]

# Obtener el top-5 usando FAISS
q_vec = query_embedding.astype("float32")[None, :]
similarities, indices = index_a_ip.search(q_vec, 5)
similarities = similarities[0]
top_indices = indices[0]

print("=== EVIDENCIA DE CÁLCULO INTERNO (CÓDIGO MÍNIMO SUGERIDO) ===")
print("Query:", query)
print("Embedding shape:", query_embedding.shape)
print("Top 5 similitudes:", similarities[:5])
for idx, score in zip(top_indices, similarities):
    print(f"Doc {idx}: score={score:.4f} | {corpus[idx][:100]}")

print("\n=== COMPARACIÓN DIRECTA DE RANKINGS ===")
print("BM25 Ranking:")
for r in search_bm25(query, k=3):
    print(f"  Doc {r['index']} (score={r['score']:.4f}): {r['text']}...")

print("\nDense B (MiniLM) Coseno Ranking:")
for r in search_dense_b(query, "cosine", k=3):
    print(f"  Doc {r['index']} (score={r['score']:.4f}): {r['text']}...")

=== EVIDENCIA DE CÁLCULO INTERNO (CÓDIGO MÍNIMO SUGERIDO) ===
Query: How accurate is the rotating black hole Gargantua physics?
Embedding shape: (384,)
Top 5 similitudes: [0.74688745 0.63255125 0.59562993 0.55208087 0.5140465 ]
Doc 3: score=0.7469 | Gargantua is the massive, rapidly rotating black hole in Interstellar. Kip Thorne's equations were u
Doc 1: score=0.6326 | The movie's scientific advisor, theoretical physicist Kip Thorne, worked on the physics of the black
Doc 12: score=0.5956 | The tesseract, or five-dimensional space, is constructed by future humans inside Gargantua. It allow
Doc 7: score=0.5521 | Plan A, developed by Professor Brand, involves solving a gravity equation to launch massive space st
Doc 4: score=0.5140 | The emotional core of Interstellar revolves around the relationship between Cooper and his daughter 

=== COMPARACIÓN DIRECTA DE RANKINGS ===
BM25 Ranking:
  Doc 1 (score=8.6652): The movie's scientific advisor, theoretical physicist Kip Thorne, worked on t

In [ ]:
# COMPARACIÓN SEMÁNTICA DE MODELOS
query = "emotional bond of parent and child"
chosen_model = "A"

if chosen_model == "A":
    results = search_dense_a(query, metric="cosine", k=3)
else:
    results = search_dense_b(query, metric="cosine", k=3)

print(f"=== RESULTADOS CON MODELO {chosen_model} ===")
for r in results:
    print(f"  Doc {r['index']} (sim={r['score']:.4f}): {r['text']}...")

=== RESULTADOS CON MODELO A ===
  Doc 4 (sim=0.6818): The emotional core of Interstellar revolves around the relationship between Cooper and his daughter Murph. Cooper's promise to return and the effects of gravitational time dilation shape their bond....
  Doc 13 (sim=0.5377): Interstellar's soundtrack was composed by Hans Zimmer. It features a pipe organ and atmospheric melodies that emphasize the vastness of space and the intimacy of the father-daughter relationship....
  Doc 6 (sim=0.5241): The dust storms and agricultural blight on Earth have devastated food supplies. Corn is the last remaining crop, and humanity faces extinction within a generation if a new planet is not colonized....


## **2. Preguntas Avanzadas Obligatorias (Específicas del Proyecto)**

#### **1. ¿Por qué embeddings de oraciones cortas pueden capturar significado incluso sin contexto extenso?**
Los modelos de embeddings modernos (como `SentenceTransformers` basados en BERT/RoBERTa) se preentrenan con miles de millones de pares de oraciones estructuradas bajo objetivos como la pérdida de entropía cruzada contrastiva (por ejemplo, MultipleNegativesRankingLoss). Esto les permite mapear construcciones gramaticales cortas directamente en un espacio vectorial semántico donde palabras y conceptos relacionados semánticamente están espacialmente próximos. Las representaciones vectoriales densas no buscan coincidencias literales de términos (como BM25), sino que "condensan" el conocimiento léxico y estructural en un vector representativo de la oración entera.

#### **2. ¿Qué limitación tiene la similitud coseno cuando los embeddings no están normalizados?**
La similitud de coseno calcula la relación del producto punto escalado por las normas de ambos vectores: $cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \|\vec{B}\|}$. 
Si los embeddings **no están normalizados** y utilizamos una función de búsqueda que dependa únicamente del producto punto directo (como `IndexFlatIP` de FAISS sin normalizar), la magnitud de los vectores (norma $L_2$) dominará el cálculo. Un vector con componentes numéricamente grandes puede sesgar el score y salir primero en el ranking, aunque su dirección angular sea diferente a la consulta. Para corregir esto, se normalizan los embeddings a norma unitaria (norma $1.0$), en cuyo caso el producto punto simple de FAISS es algebraicamente idéntico a la similitud coseno, eliminando el coste de la división en tiempo de inferencia.

#### **3. ¿Cómo afecta el tamaño del vocabulario del modelo de embeddings a la calidad de la representación?**
El tamaño del vocabulario de tokens determina la granularidad y la cobertura del modelo:
- Si el vocabulario es **demasiado pequeño**, el tokenizador se verá obligado a dividir muchas palabras en sub-tokens extremadamente pequeños (o usar tokens desconocidos `<UNK>`), lo que rompe la coherencia morfológica y diluye la representación semántica final.
- Si el vocabulario es **demasiado grande**, el modelo requiere muchos más parámetros para la matriz de embeddings de tokens, lo que aumenta la huella de memoria y puede inducir a un sobreajuste en palabras poco frecuentes.
Un buen vocabulario (e.g., 30k tokens en WordPiece/BPE) permite representar términos técnicos y coloquiales con una granularidad equilibrada.

#### **4. ¿Por qué una búsqueda semántica puede devolver resultados irrelevantes que son numéricamente similares?**
Este fenómeno ocurre por el sesgo espacial o "alucinación geométrica" del modelo. Los modelos densos mapean textos a un espacio continuo. Si una consulta contiene términos que comparten estructura formal o relaciones contextuales amplias en el conjunto de entrenamiento (pero que son lógicamente falsas para la consulta específica), el modelo puede proyectar ambos vectores muy cerca en el espacio de alta dimensión. Por ejemplo, la consulta "Planeta habitado" y el texto "Planeta helado e inhóspito" pueden compartir alta similitud coseno porque ambos hablan de características planetarias espaciales extremas, a pesar de tener significados opuestos en cuanto a habitabilidad.

#### **5. ¿Qué diferencia hay entre un embedding de promedio de tokens y un embedding de pooling especializado?**
- **Promedio de Tokens (Mean Pooling)**: Promedia aritméticamente los embeddings de todos los tokens de la última capa oculta de la red neuronal. Tiende a suavizar la representación y dar el mismo peso a todas las palabras (incluso conectores).
- **Pooling Especializado (e.g., CLS-Pooling)**: Utiliza la representación del token de clasificación especial `[CLS]` al inicio de la oración, el cual ha acumulado información de toda la secuencia mediante las capas de auto-atención del Transformer. Modelos afinados bajo arquitecturas siamesas suelen estar optimizados específicamente para `Mean Pooling` o pooling de atención, capturando dependencias estructurales más finas que un simple promedio.

## **3. Preguntas Transversales Obligatorias**

#### **1. ¿Qué parte de tu trabajo corresponde a retrieval, qué parte a generación y qué parte a razonamiento del agente?**
En este notebook nos centramos de forma pura en la fase de **Retrieval**: la ingesta del corpus, el cálculo de representaciones densas y esparzas, la indexación eficiente (FAISS y BM25) y la fusión híbrida de puntajes. No incluimos generación (LLM paramétrico) ni razonamiento del agente (LangGraph/LangChain) con el fin de realizar un análisis exhaustivo y aislar las variables del buscador, eliminando el ruido y la estocasticidad de la respuesta de un modelo de generación.

#### **2. ¿Qué componente de tu notebook sería más difícil de detectar si estuviera mal implementado?**
La **normalización de scores híbridos**. Si la función `min_max_normalize` tuviera un error (por ejemplo, no aplicar correctamente el escalado o alinear incorrectamente los índices de los scores léxicos con los densos), el código seguiría ejecutándose sin errores de sintaxis y devolvería documentos, pero los puntajes fusionados estarían distorsionados. Esto provocaría que uno de los dos retrievers (BM25 o denso) dominara completamente la búsqueda de forma silenciosa, degradando la calidad del retrieval híbrido sin lanzar una excepción de Python.

#### **3. ¿Qué resultado podría parecer bueno, pero ser técnicamente engañoso?**
Un **Recall@3 de 1.0 en consultas extremadamente específicas**. Si evaluamos el sistema únicamente con consultas que contienen palabras exactas del corpus, el Recall será perfecto. Sin embargo, esto es engañoso porque solo demuestra la memorización del sistema (overfitting de pruebas) y no mide la capacidad de generalización o comprensión conceptual ante sinónimos o consultas redactadas por usuarios reales con variaciones sintácticas.

#### **4. ¿Qué variable cambiaste y qué variable mantuviste constante para que la comparación sea justa?**
- **Constantes**: El corpus de 15 documentos, las 5 consultas de evaluación, la métrica de similitud para comparar el modelo (Coseno en FAISS) y la dimensión de salida de los vectores ($d=384$ para ambos modelos).
- **Variables**: El modelo de embeddings (`BAAI/bge-small-en-v1.5` vs `sentence-transformers/all-MiniLM-L6-v2`) y la métrica geométrica en FAISS ($L_2$ sin normalizar vs. Coseno con normalización).

#### **5. ¿Qué parte de tu resultado depende del corpus y no del modelo?**
El comportamiento del **BM25**. La distribución y los scores asignados por BM25 dependen enteramente del corpus, su vocabulario y la frecuencia de términos en los 15 documentos (frecuencia documental inversa, IDF). Si el corpus cambiara a otro dominio, las métricas léxicas se verían fuertemente alteradas, mientras que el modelo de embeddings mantendría su capacidad inherente de mapear conceptos al espacio latente preentrenado.

## **4. Puente al Curso y Conclusión Técnica**

### **Qué hice, por qué lo hice y qué significan mis resultados**
- **Qué hice**: Construí un pipeline de búsqueda híbrida que integra un modelo léxico BM25 con dos modelos de embeddings densos de la misma dimensión ($d=384$): la línea base `BGE-small` y la variante `all-MiniLM-L6-v2` usando FAISS.
- **Por qué lo hice**: Para evaluar cómo impacta la arquitectura interna y el entrenamiento del modelo de embeddings en la calidad final de recuperación (medida mediante Recall@3 y MRR@3), aislando la dimensión para realizar una comparación justa.
- **Qué significan los resultados**: Las pruebas cuantitativas demuestran que el **retriever híbrido** supera consistentemente a los modelos léxicos y densos individuales. El modelo `BGE-small` arrojó una similitud semántica superior a `all-MiniLM-L6-v2` en consultas que involucraban paráfrasis semánticas complejas (como la relación familiar de Cooper/Murph), validando su optimización específica para tareas de recuperación de información.

### **Puente al Curso**
Este proyecto se conecta directamente con:
1. **Búsqueda Semántica**: Concepto clave estudiado en el curso para evitar los límites de coincidencia exacta de palabras y capturar intenciones a través de proyecciones vectoriales.
2. **Evaluación de Retrieval**: Implementación de métricas de ranking clásicas (Recall@k, MRR) para auditar la calidad del contexto recuperado en una arquitectura RAG real.

## **Declaración de Autoría y Uso de IA**

```
Declaro que comprendo el código, los resultados y las explicaciones entregadas en esta Práctica.
Si utilicé herramientas de IA, las usé como apoyo para redacción, depuración o consulta, pero la implementación final, la interpretación técnica y la defensa del trabajo son responsabilidad mía.
```

**Detalle del apoyo de IA:**
- Se utilizó IA como apoyo para redactar de forma clara las respuestas a las preguntas avanzadas específicas y transversales.
- Se utilizó IA para estructurar el formato JSON del notebook de manera limpia y depurar la alineación de scores de FAISS en la búsqueda híbrida.